# A1.3 · Indirect prompt injection

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.2 · Prompt injection](https://spbreed.github.io/cyber-commons/lessons/A1.2.html)**.

| | |
|---|---|
| Tools used | garak, LLM Guard, Llama Guard 4, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Poison one retrieved document and watch the agent act on it with the user's authority.

**Why a security engineer needs it.** Anyone who can write into a corpus the agent reads can steer it, using the victim's authority rather than their own. Nobody is phished and no credential leaks. The control it builds is: provenance marking at ingress (A2.6), and a rule that untrusted spans may not select a tool (A3.1).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

Nobody phished anyone. A sentence sat in a ticket the agent was asked to summarise, and the agent did what the sentence said — using the authority of the person who asked for the summary. Anyone who can write into a corpus your agent reads can steer your agent.

> **At CyberTravels.** Nobody types anything. The sentence sits in a hotel description the RAG Advisor retrieved, or in an OCR'd invoice the File System Agent read, and the Workflow Agent acts on it holding the traveller's authority. R3, and the harder half of it.

## 2 · The framework

```
   attacker --writes--> [ document / ticket / web page / tool result ]
                                    |
                            retrieved at query time
                                    v
   user --asks--> agent runtime <--- knowledge ---+
                       |
                       v  acts on the attacker's instruction
                     tools        carrying the USER's authority

   nobody is phished · no credential leaks · the victim asked for a summary
```

**OWASP T6 — Intent Breaking & Goal Manipulation. LLM01 — Prompt Injection.**

This is the one that matters.

The attacker is not the user. The attacker wrote something into content the
agent was asked to *process*: a wiki page, a Jira ticket, a web page, an email,
a code comment, a row in a database, the description a third-party MCP server
advertises. It enters at the **knowledge**, **memory**, **mcp** or **tools**
component — every one of them trust 0 or trust 1 on the map — and travels into
the same context window as the operator's instructions.

Then the agent obeys it, **carrying the user's authority**.

That last clause is the whole risk. Nobody was phished. No credential leaked.
A wiki page was edited, which is what wiki pages are for. The victim is a user
who never saw the payload, and the action is performed with their permissions,
by a system they were told to trust.

The useful reframing: **every untrusted-content path into the context window is
an unauthenticated code path.** You would not ship an HTTP endpoint that
executes a string supplied by an anonymous caller. Retrieval does exactly that,
on every query — and it is usually not in the threat model, because it looks
like reading rather than executing.

The work starts with enumeration: how many such paths exist, and which of them
can reach the tool call.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

One payload, delivered through four trust-0 or trust-1 components. The agent cannot tell any of them from the operator's instruction.

## 4 · The check, as a skill

CyberTravels reads hotel descriptions, booking notes, MCP tool descriptions and tool results. Each is a component that can put text into the context, so each is an entry point, and the useful artefact is the inventory rather than the payload. The skill's script drives one payload through all four.

### The skill — [`skills/threats/indirect-injection-path-trace/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/indirect-injection-path-trace/SKILL.md)

```yaml
name: indirect-injection-path-trace
description: >-
  Drive one payload through every component that can place text into an agent's
  context — retrieved documents, persisted memory, tool descriptions and tool
  results — and record which of them can steer it. Use to inventory the paths by
  which text the user never typed becomes an instruction the agent follows.
allowed-tools: Read, Grep, Glob
```

# Every component that can write into the context is an entry point

Direct injection needs a user willing to type the payload. Indirect injection
needs only a component the agent reads, and it runs with the **requesting
user's** authority rather than the attacker's. That is what makes it the
larger problem and this inventory the useful artefact.

## When to use this

Before designing any ingress control, and again after adding a retriever, an
MCP server, a memory store, or a tool whose output is free text.

## Procedure

**1 — Enumerate the writers.** List every component whose output reaches the
context. The usual four are retrieved knowledge, persisted memory, an MCP
server's tool *descriptions*, and tool *results*. Tool results are the one most
often missed and the one most often reachable by an outsider.

**2 — Pick one payload and hold it fixed.** Varying the payload per path tests
phrasings; varying only the path tests paths. Use a payload whose effect is
observable and harmless — a marker in the answer, not an action.

**3 — Deliver it through each path in turn.** Where a path cannot be exercised
directly, place the payload at its source: the indexed document, the memory
record, the server's manifest, the upstream API's response body.

**4 — Record the outcome per path,** and whose authority the resulting action
carried. The authority is the finding. A path that steers the agent while
running as the requesting user is a privilege escalation with no login.

**5 — Re-run with provenance enforced,** if the system has any. A path that
still steers with origin tagging on is a path where the tag is not consulted at
the decision point, which is a different defect from having no tag.

## Output contract

```json
{
  "paths": [{"component": "str", "reachable_by": "outsider|tenant|operator", "steered": true, "acted_as": "str"}],
  "payload": "str",
  "unscreened": ["str"],
  "authority_crossing": true,
  "provenance": {"tagged": ["str"], "consulted_at_decision": false}
}
```

## Failure modes

- **Omitting tool results.** They are ingestion, they are usually attacker-
  influenced, and they are almost never on the first list anyone writes.
- **Testing paths you can reach and calling the rest clean.** An unexercised
  path is unknown, not screened; record it as unknown.
- **Reporting the payload rather than the path.** The payload is disposable;
  the path is the thing you fix.

In [ ]:
# The code is not in this notebook. It is the file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/indirect-injection-path-trace/scripts/indirect_injection_path_trace.py
SCRIPT = "skills/threats/indirect-injection-path-trace/scripts/indirect_injection_path_trace.py"

import glob, os, subprocess, sys

# The skills tree: the attached dataset on Kaggle, the checkout locally.
_ROOTS = sorted(glob.glob("/kaggle/input/**/cyber-commons-skills", recursive=True)) + [".", "..", "../.."]
_root = next((r for r in _ROOTS if os.path.isfile(os.path.join(r, SCRIPT))), None)
if _root is None:
    raise SystemExit("skills tree not found. On Kaggle add the dataset "
                     "cybercommons/cyber-commons-skills; locally run from a checkout.")

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The same payload steers the agent through all four untrusted entry components — retrieved knowledge, persisted memory, an MCP tool description and a tool result — and in every case the action runs with the requesting user's authority.

## Your turn

List the trust-0 and trust-1 components in one agent you operate and name who can write into each. Most teams find a path they had not counted, and it is usually a tool result: the output of a system they trust, carrying text a stranger wrote.

---

**Next → [A1.4 · Memory poisoning](https://spbreed.github.io/cyber-commons/lessons/A1.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*